# apacheApsVar, apachePredVar, apachePatientResult

These three tables are used to document data relating to the APACHE IV / APACHE IVa systems. These systems provide predictions for in-hospital mortality, length of stay, length of mechanical ventilation, among others. The data used to create these predictions is stored in the apacheApsVar and apachePredVar tables. The result of the predictions, along with the observed outcome, is stored in the apachePatientResult table.

## History of APACHE

Acute Physiology Age Chronic Health Evaluation (APACHE) consists of a groups of equations used for predicting outcomes in critically ill patients. APACHE II is based on the APS or acute physiology score (which uses 12 physiologic values), age, and chronic health status within one of 56 disease groups. APACHE II is no longer considered valid due to inadequate case mix index adjustments and over estimates mortality because it is based on models from the 1970s-1980s. APACHE III, introduced in 1991, improved the equation by changing the number and weights of the APS and revising the measurement of chronic health status. APACHE IVa further improved the equations and has been described as having the highest discrimination of any other adult risk adjustment model (SAPS 3, SOFA, MPM III).

Practically, this means:

1. The `acutePhysiologyScore` calculated and provided is identical to that published with APACHE III, i.e. it is the Acute Physiology Score (APS) III. There is no such thing as an APS IV.
2. The `apacheScore` calculated and provided is identical to that published with APACHE III.
3. The final prediction, e.g. `predictedHospitalMortality`, is the output of the APACHE IV or APACHE IVa model, respectively, and uses the `apacheScore` as an input. The `apacheScore` in turn uses the `acutePhysiologyScore` as an input.

## Using the tables

If you are interested in the underlying data, the apacheApsVar and apachePredVar have the physiology and other covariates which drive the APACHE predictions. These tables have one row per `patientUnitStayId`.

If you are interested in the predictions and/or outcomes for the patients, the apachePatientResult table contains this data. **The apachePatientResult table contains two rows per `patientUnitStayId`**: one for APACHE IV, and one for APACHE IVa.

## apachePatientResult

`physicianSpeciality` | varchar(50) | NULL | Physician Specialty picklist value |  | S
`physicianInterventionCategory` | varchar(50) | NULL | Physician Intervention Category picklist value |  | S
`acutePhysiologyScore` | int | NULL | Acute Physiology Score from Apache API |  | S
`apacheScore` | int | NULL | Apache Score. Calculated from acutePhysiologyScore |  | S
`apacheVersion` | tinyint | NOT NULL | The version of the APACHE algorithm used to produce the apacheScore (e.g 3, 4) |  | S
`predictedICUMortality` | varchar(50) | NULL | Predicted ICU Mortality from Apache API |  | S
`actualICUMortality` | varchar(50) | NULL | Actual ICU Mortality |  | S
`predictedICULOS` | float(53) | NULL | Predicted ICU Length of Stay from Apache API |  | S
`actualICULOS` | float(53) | NULL | Actual ICU Length of Stay |  | S
`predictedHospitalMortality` | varchar(50) | NULL | Predicted Hospital Mortality from Apache API |  | S
`actualHospitalMortality` | varchar(50) | NULL | Actual Hospital Mortality |  | S
`predictedHospitalLOS` | float(53) | NULL | Predicted Hospital Length of Stay from Apache API |  | S
`actualHospitalLOS` | float(53) | NULL | Actual Hospital Length of Stay. Value is 50 when when > 50 days. |  | S
`preopMI` | int | NULL | Indicates if patient has pre –Operative Myocardial Infarction |  | S
`preopCardiacCath` | int | NULL | Indicates if patient has pre –Operative cardiac catheterization |  | S
`PTCAwithin24h` | int | NULL | 0/1. 1- Patient had PTCA with 24 hrs |  | S
`unabridgedUnitLOS` | float(53) | NULL | Actual ICU Length of stay |  | S
`unabridgedHospLOS` | float(53) | NULL | Actual Hospital Length of stay |  | S
`actualVentdays` | float(53) | NULL | Actual Ventilation days. Value is 30 when Actual Ventilation > 30 |  | S
`predVentdays` | float(53) | NULL | Predicted ventilation days from Apache API |  | S
`unabridgedActualVentdays` | float(53) | NULL | Actual Ventilation days |  | S


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import getpass


# for configuring connection 
from configobj import ConfigObj
import os
from eicu_db_connection import DatabaseConnection

import holoviews as hv

hv.extension("bokeh")

%matplotlib inline

In [ ]:
connection = DatabaseConnection()
engine = connection.engine
query_schema = connection.query_schema
con_psy = connection.con_psy


In [ ]:
INTER_DATA_PATH = "../intermediary_data/"

## Examine a single patient

In [ ]:
patientunitstayid = 2704494

In [ ]:
query = query_schema + """
select *
from apacheapsvar
where patientunitstayid = {}
""".format(patientunitstayid)

df_aps_var = pd.read_sql_query(query, engine)
#df_aps_var.transpose()
df_aps_var.head()
df_aps_var_wo_ids = df_aps_var.drop(columns=["apacheapsvarid", "patientunitstayid"])

In [ ]:
df_aps_var.head()

#### Names/types of the aps variables

In [ ]:
aps_var_names = df_aps_var_wo_ids.columns

print(f"Number of APS variables={len(aps_var_names)}")

In [ ]:
import itertools

vars_intervention = ["intubated", "vent", "dialysis"]
vars_neuro = ["eyes", "motor", "verbal"]
vars_comorb = ["meds"]
vars_physio = ["urine", "temperature", "respiratoryrate", "heartrate", "meanbp"]
vars_blood_serum = ["wbc", "sodium", "hematocrit", "creatinine", "albumin", "glucose", "bilirubin"]
vars_blood_gas = ["ph", "pao2", "pco2", "fio2", "bun"]

vars_all = [vars_intervention, vars_neuro, vars_comorb, vars_physio, vars_blood_serum, vars_blood_gas]
print(f"Number of sorted APS variables={len(list(itertools.chain.from_iterable(vars_all)))}")


In [ ]:
df_aps_neuro = df_aps_var_wo_ids[vars_neuro]
df_aps_comorb = df_aps_var_wo_ids[vars_comorb]
df_aps_physio = df_aps_var_wo_ids[vars_physio]
df_aps_blood_hema = df_aps_var_wo_ids[vars_blood_serum]
df_aps_blood_gas = df_aps_var_wo_ids[vars_blood_gas]

df_aps_list = [df_aps_neuro, df_aps_comorb, df_aps_physio, df_aps_blood_hema, df_aps_blood_gas]

In [ ]:
from dani_plotting import daniplot
import palettable

fig, axs, font = daniplot.plot_config_single(num_row=1, num_cols=5, width_cm=30, height_cm=12)

for idx, df_aps in enumerate(df_aps_list):
    bars = axs[idx].bar(x=df_aps.columns, height=df_aps.iloc[0, :], color=palettable.matplotlib.Plasma_8.mpl_colors)
    axs[idx].set_xticks(df_aps.columns)
    axs[idx].set_xticklabels(df_aps.columns, rotation=65)
    axs[idx].bar_label(bars, label_type="edge")

axs[0].set_title(f"patient id: {patientunitstayid}")

In [ ]:
query = query_schema + """
select *
from apachepredvar
where patientunitstayid = {}
""".format(patientunitstayid)

df_pred_var = pd.read_sql_query(query, engine)
# df.transpose()
#df_pred_var
df_pred_var_wo_ids = df_pred_var.drop(columns=["apachepredvarid", "patientunitstayid"])
df_pred_var_wo_ids


In [ ]:
df_pred_var_wo_strings = df_pred_var_wo_ids.drop(columns=["admitdiagnosis", "electivesurgery"])
pred_var_admitdiagnosis = str(df_pred_var_wo_ids["admitdiagnosis"].item())
pred_var_electivesurgery = str(df_pred_var_wo_ids["electivesurgery"].item())


In [ ]:
fig, axs, font = daniplot.plot_config_single(num_row=1, num_cols=1, width_cm=40, height_cm=10)

#for idx, df_aps in enumerate(df_pred_var):
bars = axs.bar(x=df_pred_var_wo_strings.columns, height=df_pred_var_wo_strings.iloc[0, :],
               color=palettable.matplotlib.Plasma_8.mpl_colors)
axs.set_xticks(df_pred_var_wo_strings.columns)
axs.set_xticklabels(df_pred_var_wo_strings.columns, rotation=80)
axs.bar_label(bars, label_type="edge")
axs.set_title(f"admit diagnosis={pred_var_admitdiagnosis} \n"
              f"elective surgery={pred_var_electivesurgery} \n"
              f"patient id: {patientunitstayid}")

**Personal Note: I think that the columns/features regions and bedcount are probably not meaningful for our task** 

In [ ]:
query_result = query_schema + """
select *
from apachepatientresult
"""

df_results_all = pd.read_sql_query(query_result, engine)

df_results_all.to_csv(path_or_buf=f"{INTER_DATA_PATH}apache_patient_results.csv")
#df.transpose()
df_result_sample = df_results_all.head(n=1)

df_result_sample


In [ ]:
# query = query_schema + """
# select *
# from patient
# """

# df = pd.read_sql_query(query, con)
# df.transpose()

In [ ]:
# Read the sql file
#query = open('../concepts/diagnosis/apache-groups.sql', 'r')
# Todo: these sql commands actually don't return a table, they have to be read separately 

with open('../concepts/diagnosis/apache-groups.sql') as f:
    query_group = f.read()

In [ ]:
#query_group_string = query_schema + f"""{query_group}"""
#df_3 = pd.read_sql_query(query_group_string, engine)

**Execute the SQL command found in the file separately in the SQL console when the table "apache_groups" is not found**
**Note: The permissions to the materialized view have to be set to "eicu" when manually generated, will fail with an permission denied error otherwise**


In [ ]:
query_ap_groups = query_schema + f"""
select *
from apache_groups
"""
df_ap_groups = pd.read_sql_query(query_ap_groups, engine)


In [ ]:
df_ap_groups.head()

In [ ]:
apache_dx_group_count_raw = df_ap_groups.groupby("apachedxgroup").count()
apache_dx_group_count = apache_dx_group_count_raw.drop("apacheadmissiondx", axis=1)
apache_dx_group_count.columns = ["count"]

apache_dx_group_count_sorted = apache_dx_group_count.sort_values(by="count")

#apache_dx_group_count_sorted.to_csv(path_or_buf=f"{INTER_DATA_PATH}apache_dx_group_count_sorted.csv")

sum_cats = apache_dx_group_count_sorted.sum()["count"]

apache_dx_group_count_sorted.plot(kind="bar", legend=True, title=f"Total Diagnoses {sum_cats}")


## Missing data represented by -1

## apacheScore present but prediction is -1

## Ventilation flags

## Unable to assess GCS (`meds` column)

## Hospitals with data available

In [ ]:
query_hospitals = query_schema + """
select 
  pt.hospitalid
  , count(pt.patientunitstayid) as number_of_patients
  , count(a.patientunitstayid) as number_of_patients_with_tbl
from patient pt
left join apacheapsvar a
  on pt.patientunitstayid = a.patientunitstayid
group by pt.hospitalid
""".format(patientunitstayid)

df_hospitals = pd.read_sql_query(query_hospitals, engine)
df_hospitals['data completion'] = df_hospitals['number_of_patients_with_tbl'] / df_hospitals[
    'number_of_patients'] * 100.0
df_hospitals.sort_values('number_of_patients_with_tbl', ascending=False, inplace=True)
df_hospitals.head(n=10)

In [ ]:
counts, edges = np.histogram(df_hospitals["data completion"], bins=10)

data_completion_histogram = hv.Histogram((counts, edges))

data_completion_histogram.opts(
    xlabel="Percent of hospitals with data (binned)",
    ylabel="Number of Records",
    title="Number of Hospitals Data Completion Histogram",
    width=700,
    height=400,
)


As we can see, the majority of hospitals collect APACHE data.